# 🧹 Phase 1: Data Cleaning & Pre-processing

This notebook implements a high-performance, **parallelized cleaning pipeline** designed to handle large anime datasets efficiently.

### 🚀 Key Optimization Strategies
-   **Atomic Pipelines**: Each dataset is processed by a specialized, modular pipeline.
-   **Parallel Processing**: CPU-intensive tasks are distributed across all available cores.
-   **Memory Management**: Aggressive downcasting and categorical conversion typically reduce memory usage by **80%+**.
-   **Referential Integrity**: A progressive `GLOBAL_ID_MAP` ensures all datasets link to valid, canonical IDs.


In [14]:
%load_ext autoreload
%autoreload 2

import gc
import os
from joblib import Parallel, delayed
import pandas as pd

# 🧪 Standard Imports
from solution.src.common import PROCESSED_DATA_DIR
from solution.src.data import load_raw, get_raw_path, PROCESSED_FILES
from solution.src.utils import save_refined, print_fancy_progress

# 📦 Cleaning Utilities
from nb_01_cleaning import (
    REMAINING_FILES,
    standardize_column_names_and_text,
    trim_all_text_columns,
    merge_content_duplicates,
    remove_orphaned_records,
    get_memory_usage_mb,
    reduce_memory_usage,
    display_cleaning_summary,
    display_foreign_key_merges,
    process_ratings_batch,
    clean_and_optimize_dataset,
    count_total_rows_in_file,
    clean_characters_dataset,
    clean_anime_details_dataset,
    clean_anime_stats_dataset
)

# Ensure processed directory exists
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
print(f"Saving to:    {PROCESSED_DATA_DIR}")

# Global ID Map (relational integrity)
GLOBAL_ID_MAP = {}


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Saving to:    C:\Users\chrst\repos\unito-hci-data-analysis-25-26\data\processed


## 1. Characters (`characters.csv`)

**Role**: Establishes the valid universe of Character IDs.

**Processing Steps**:
-   **Deduplication**: Identifies and merges duplicate characters, updating the Global ID Map.
-   **Cleaning**: Drops `about` sections to save space (they're not important for our needs).
-   **Validation**: Captures `valid_character_ids` to filter orphan references in other files.

In [15]:
# 1. Load
df_chars = load_raw('characters')

# 2. Clean Pipeline
df_chars, id_map, report = clean_characters_dataset(df_chars)

# 3. Update Global Knowledge
if id_map:
    GLOBAL_ID_MAP['character_id'] = id_map

# 4. Report
display_cleaning_summary("Characters", report)

# 5. Optimize & Save
df_chars = reduce_memory_usage(df_chars)
save_refined(df_chars, PROCESSED_DATA_DIR / "characters.parquet")

# 6. Context Capture
valid_character_ids = set(df_chars['character_id'].dropna())
print(f"Captured {len(valid_character_ids)} valid Character IDs.")

gc.collect();


Loading characters.csv with PyArrow...
--- Characters Summary ---
  Initial Rows: 209963
  Standardization: {'duplicates_dropped': 456}
  Content Resolution: {'duplicates_found': 0, 'duplicates_dropped': 0, 'id_map': {}}
  Final Rows: 209507
Memory optimized: 74.48 MB -> 73.28 MB (1.6% reduction)
Saving to C:\Users\chrst\repos\unito-hci-data-analysis-25-26\data\processed\characters.parquet (parquet)...
Save complete.
Captured 209506 valid Character IDs.


## 2. Anime Details (`details.csv`)

**Role**: Establishes the valid universe of Anime IDs.

**Processing Steps**:
-   **Imputation**: Derives missing `year` from `start_date`.
-   **Type Conversion**: Parses dates and converts low-cardinality columns (Genres, Studios) to Categories.
-   **Validation**: Captures `valid_anime_ids`.

In [16]:
# 1. Load
df_details = load_raw('details')

# 2. Clean Pipeline
df_details, id_map, report = clean_anime_details_dataset(df_details)

# 3. Update Global Knowledge
if id_map:
    GLOBAL_ID_MAP['anime_id'] = id_map

# 4. Report
display_cleaning_summary("Details", report)

# 5. Optimize & Save
df_details = reduce_memory_usage(df_details)
save_refined(df_details, PROCESSED_DATA_DIR / "details.parquet")

# 6. Context Capture
valid_anime_ids = set(df_details['anime_id'].dropna())
print(f"Captured {len(valid_anime_ids)} valid Anime IDs.")

gc.collect();


Loading details.csv with PyArrow...
--- Details Summary ---
  Initial Rows: 28955
  Standardization: {'duplicates_dropped': 0}
  Content Resolution: {'duplicates_found': 0, 'duplicates_dropped': 0, 'id_map': {}}
  Final Rows: 28955
  Year Imputed: 21838
Memory optimized: 42.17 MB -> 29.94 MB (29.0% reduction)
Saving to C:\Users\chrst\repos\unito-hci-data-analysis-25-26\data\processed\details.parquet (parquet)...
Save complete.
Captured 28955 valid Anime IDs.


## 3. Anime Stats (`stats.csv`)

**Role**: Supplementary validation source. Often contains duplicates not present in `details.csv`.

**Processing Steps**:
-   **Merge Duplicates**: Updates the `GLOBAL_ID_MAP` if new duplicates are found.
-   **Referential Integrity**: Ensures all entries map to valid Anime IDs from Step 2.

In [17]:
# 1. Load
df_stats = load_raw('stats')

# 2. Clean Pipeline
df_stats, id_map, report = clean_anime_stats_dataset(df_stats)

# 3. Update Global Knowledge (Merge if new duplicates found)
if id_map:
    if 'anime_id' not in GLOBAL_ID_MAP:
        GLOBAL_ID_MAP['anime_id'] = {}
    GLOBAL_ID_MAP['anime_id'].update(id_map)

# 4. Report
display_cleaning_summary("Stats", report)

# 5. Optimize & Save
df_stats = reduce_memory_usage(df_stats)
save_refined(df_stats, PROCESSED_DATA_DIR / "stats.parquet")

gc.collect();


Loading stats.csv with PyArrow...
--- Stats Summary ---
  Initial Rows: 28955
  Standardization: {'duplicates_dropped': 0}
  Content Resolution: {'duplicates_found': 30, 'duplicates_dropped': 16, 'id_map': {np.float64(61643.0): np.float64(61635.0), np.float64(62211.0): np.float64(61653.0), np.float64(62539.0): np.float64(61659.0), np.float64(62274.0): np.float64(61626.0), np.float64(62443.0): np.float64(61619.0), np.float64(61646.0): np.float64(61563.0), np.float64(60569.0): np.float64(58230.0), np.float64(60579.0): np.float64(58230.0), np.float64(45746.0): np.float64(45734.0), np.float64(47151.0): np.float64(47150.0), np.float64(47384.0): np.float64(47383.0), np.float64(47138.0): np.float64(46881.0), np.float64(62558.0): np.float64(62557.0), np.float64(62559.0): np.float64(62557.0), np.float64(60405.0): np.float64(60392.0), np.float64(44117.0): np.float64(44115.0)}}
  Final Rows: 28939
Memory optimized: 6.18 MB -> 3.48 MB (43.8% reduction)
Saving to C:\Users\chrst\repos\unito-hci-data

## 4. User Ratings (`ratings.csv`)

**Challenge**: Huge dataset (>120M rows). Cannot process in memory sequentially without being slow.

**Solution: Parallel Streaming Approach**
-   **Parallel Workers**: The file is read in chunks, and each chunk is processed by a separate CPU core.
-   **Streaming Write**: Processed chunks are immediately written to disk (Parquet), keeping RAM usage low.
-   **FK Remapping**: We apply the `GLOBAL_ID_MAP` here to update Anime IDs that were merged in previous steps. 
-   **Orphan Retention**: Votes for non-existent Anime are **retained** (per user request).
-   **Deduplication Strategy**: **SKIPPED**. Analysis of a 15M row sample showed **0** duplicates. Global deduplication (sorting 120M rows) is prohibitively expensive for negligible gain.

In [18]:
chunk_size = 5_000_000
csv_path = get_raw_path('ratings')

print(f"Processing {csv_path} in parallel chunks...")

# Context for workers
id_map_anime = GLOBAL_ID_MAP.get('anime_id', {})

stats = {'rows_read': 0, 'fk_remapped': 0, 'fk_samples': []}

print("Counting rows for progress estimation...")
total_rows = count_total_rows_in_file(csv_path) - 1
estimated_total_chunks = (total_rows // chunk_size) + (1 if total_rows % chunk_size > 0 else 0)
print(f"Target: {total_rows:,} rows across {estimated_total_chunks} chunks.")

# Parallel Pipeline Generator
# load_raw will automatically return an iterator (chunks) if file is large + low RAM
reader = load_raw('ratings')

# If reader is not an iterator (i.e. we have tons of RAM), we must force it to be one for parallel proc
# or just handle it as a single chunk. But existing code wants parallel.
# For now, let's assume if it's NOT an iterator, we just wrap it in a list so it looks like one chunk.
if isinstance(reader, pd.DataFrame):
    reader = [reader]

processed_chunks = Parallel(n_jobs=-1, return_as='generator')(
    delayed(process_ratings_batch)(chunk, id_map_anime)
    for chunk in reader
)

# Streaming Write
def result_generator():
    for i, (processed_chunk, metrics) in enumerate(processed_chunks, 1):
        print_fancy_progress(i, estimated_total_chunks, f'Chunk {i}: ')

        stats['rows_read'] += metrics['rows_read']
        stats['fk_remapped'] += metrics['fk_remapped']
        if len(stats['fk_samples']) < 5:
            stats['fk_samples'].extend(metrics['fk_samples'])

        yield processed_chunk

save_refined(result_generator(), PROCESSED_DATA_DIR / "ratings.parquet")

# Final Report
final_report = {
    "Total Rows Read": stats['rows_read'],
    "FKs Remapped": stats['fk_remapped'],
    "Final Rows Est.": stats['rows_read']
}
display_cleaning_summary("Ratings", final_report)
display_foreign_key_merges(stats['fk_samples'])

gc.collect();


Processing C:\Users\chrst\repos\unito-hci-data-analysis-25-26\data\raw\ratings.csv in parallel chunks...
Counting rows for progress estimation...
Target: 124,298,357 rows across 25 chunks.
  [WARN] Low RAM detected (3.7GB) for file ratings.csv (4.5GB).
  -> Falling back to dynamic chunking (chunksize=5000000).
Loading ratings.csv with Pandas (Engine: c)...
Saving to C:\Users\chrst\repos\unito-hci-data-analysis-25-26\data\processed\ratings.parquet (parquet)...
Chunk 25:  ------------------------------ 100.0%
Save complete.
--- Ratings Summary ---
  Total Rows Read: 124298357
  FKs Remapped: 739
  Final Rows Est.: 124298357

--- Foreign Key Remapping ---
Updated references to merged IDs:
  Batch 1: 61646 -> 61563
  Batch 2: 60579 -> 58230
  Batch 3: 61643 -> 61635
  Batch 4: 61646 -> 61563
  Batch 5: 61646 -> 61563
------------------------------


## 5. Remaining Files

**Strategy**: Apply the same rigor to the remaining 9 Relational and Metadata tables.

**Parallel Execution**:
We use `clean_and_optimize_dataset` (defined in `nb_01_cleaning` package) to run the cleaning pipeline for all these files **concurrently**. 

**Understanding the Report:**
-   **`Clean` Impact**: Memory usage usually *increases* here (e.g., `-10%` reduction) because we expand efficient storage into Python objects for cleaning.
-   **`Opt` Impact**: Memory usage *decreases* massively (e.g., `+80%` reduction) as we convert to efficient types.
-   **`Total` Impact**: The net result. Usually a significant reduction, ensuring the final Parquet files are highly optimized.

In [19]:
print(f"Processing {len(REMAINING_FILES)} other files in parallel...")

results = Parallel(n_jobs=-1)(
    delayed(clean_and_optimize_dataset)(name, PROCESSED_FILES)
    for name in REMAINING_FILES
)

print(f"\n--- Detailed Batch Report ---")
for res in results:
    total_red = (res['mem_src'] - res['mem_dst']) / res['mem_src'] * 100 if res['mem_src'] > 0 else 0

    print(f"[{res['file']}]")
    print(f"  Rows:   {res['rows_src']} -> {res['rows_dst']}")
    print(f"  Memory: {res['mem_src']:.1f}MB -> {res['mem_dst']:.1f}MB ({total_red * -1:+.1f}%)")
    if res['actions']:
        print(f"  Actions: {', '.join(res['actions'])}")
    print("-" * 30)

print(f"Total files processed: {len(results)}")

gc.collect();


Processing 9 other files in parallel...

--- Detailed Batch Report ---
[person_details]
  Rows:   76699 -> 76696
  Memory: 39.6MB -> 25.3MB (-36.1%)
  Actions: Removed duplicates, Parsed birthdays
------------------------------
[profiles]
  Rows:   337155 -> 337155
  Memory: 171.5MB -> 28.5MB (-83.4%)
  Actions: Parsed birthdays, Parsed join dates
------------------------------
[favs]
  Rows:   4178747 -> 4178747
  Memory: 486.8MB -> 57.6MB (-88.2%)
------------------------------
[recommendations]
  Rows:   105249 -> 105249
  Memory: 1.6MB -> 0.8MB (-50.0%)
------------------------------
[character_anime_works]
  Rows:   236816 -> 236816
  Memory: 30.3MB -> 13.7MB (-54.8%)
------------------------------
[character_nicknames]
  Rows:   37080 -> 36978
  Memory: 2.4MB -> 2.5MB (+5.9%)
  Actions: Removed duplicates
------------------------------
[person_alternate_names]
  Rows:   20465 -> 20426
  Memory: 1.4MB -> 1.6MB (+12.2%)
  Actions: Removed duplicates
------------------------------
[